In [ ]:
from PIL import Image
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from pathlib import Path
from collections import Counter
from matplotlib import colors

In [ ]:
import re
import os
import hashlib
import numpy as np
import pandas as pd
import torch as to
import matplotlib.pyplot as plt
import torch.nn as nn

In [ ]:
DATA_DIR = Path("data")
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15
LR = 1e-4
PATIENCE = 4         
SEED = 42
DEVICE = "cuda" if to.cuda.is_available() else "cpu"
CLASSES = {"non-COVID": 0, "COVID": 1}
EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

In [ ]:
to.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
rows = []
for cls_dir in sorted(p for p in DATA_DIR.iterdir() if p.is_dir()):
    for p in cls_dir.rglob("*"):
        if p.suffix.lower() not in EXTS:
            continue
        row = {"path": str(p), "cls": cls_dir.name, "name": p.name, "ok": True}
        try:
            with Image.open(p) as im:
                im.verify()                     
            with Image.open(p) as im:
                arr = np.asarray(im.convert("L"), dtype=np.float32)
                row.update(
                    width=im.width, height=im.height, mode=im.mode,
                    mean=arr.mean(), std=arr.std(),
                    min=arr.min(), max=arr.max(),
                    black_frac=(arr < 10).mean(),   
                )
            row["md5"] = hashlib.md5(p.read_bytes()).hexdigest()
        except Exception as e:
            row.update(ok=False, error=str(e))
        rows.append(row)
 
df = pd.DataFrame(rows)
print(f"\nTotal images found: {len(df)}")

In [ ]:
print("\n=== Class balance ===")
print(df.cls.value_counts().to_string()) # df["cls"].value_counts().to_string() also works

In [ ]:
bad = df[~df.ok]
print(f"\n=== Corrupt/unreadable files: {len(bad)} ===")
if len(bad):
    print(bad[["path", "error"]].head(10).to_string())
df = df[df.ok].copy()

In [ ]:
print("\n=== Image sizes (top 10) ===")
sizes = Counter(zip(df.width, df.height))
for (w, h), n in sizes.most_common(10):
    print(f"{w}x{h}: {n}")
print(f"Distinct sizes: {len(sizes)}")
 
print("\n=== Color modes ===")
print(df["mode"].value_counts().to_string())
 
print("\n=== Size per class (mean width x height) ===")
print(df.groupby("cls")[["width", "height"]].mean().round(0).to_string())

In [ ]:
print("\n=== Exact duplicates ===")
dups = df[df.duplicated("md5", keep=False)]
print(f"Files involved in duplicates: {len(dups)}")
cross = dups.groupby("md5").cls.nunique()
print(f"Duplicate groups with CONFLICTING labels: {(cross > 1).sum()}")
if len(dups):
    print(dups.sort_values("md5")[["md5", "cls", "name"]].head(10).to_string())
   

In [ ]:

conflicting_hashes = df.groupby("md5")["cls"].nunique()
conflicting_hashes = conflicting_hashes[conflicting_hashes > 1].index
conflicts_removed = df["md5"].isin(conflicting_hashes).sum()
df = df[~df["md5"].isin(conflicting_hashes)].copy()


before = len(df)
df = df.drop_duplicates(subset="md5", keep="first").copy()
duplicates_removed = before - len(df)

print(f"Rows excluded for conflicting labels: {conflicts_removed}")
print(f"Extra duplicate rows excluded: {duplicates_removed}")
print(f"Rows remaining: {len(df)}")

In [ ]:
print("\n=== Pixel statistics per class (differences can signal shortcuts) ===")
print(df.groupby("cls")[["mean", "std", "min", "max", "black_frac"]].mean().round(3).to_string())

In [ ]:
n = 8
classes = sorted(df.cls.unique())
fig, axes = plt.subplots(len(classes), n, figsize=(2.2 * n, 2.4 * len(classes)))
axes = np.atleast_2d(axes)
for r, cls in enumerate(classes):
    sample = df[df.cls == cls].sample(min(n, (df.cls == cls).sum()), random_state=SEED)
    for c, (_, row) in enumerate(sample.iterrows()):
        axes[r, c].imshow(Image.open(row.path).convert("L"), cmap="gray")
        axes[r, c].axis("off")
        if c == 0:
            axes[r, c].set_title(cls, fontsize=10, loc="left")
plt.tight_layout()
plt.savefig("sample_grid.png", dpi=120)
print("\nSaved sample_grid.png")
 

In [ ]:
plt.figure(figsize=(6, 3.5))
for cls in classes:
    plt.hist(df[df.cls == cls]["mean"], bins=30, alpha=0.5, label=cls)
plt.xlabel("mean pixel intensity per image"); plt.ylabel("count"); plt.legend()
plt.tight_layout(); plt.savefig("intensity_hist.png", dpi=120)
print("Saved intensity_hist.png")
 
df.drop(columns=["md5"]).to_csv("data_inventory.csv", index=False)
print("Saved data_inventory.csv")
 

In [ ]:
df = pd.read_csv("data_inventory.csv")
df = df[df.ok] if "ok" in df.columns else df
classes = sorted(df.cls.unique())
colors = dict(zip(classes, plt.cm.tab10.colors))
Path("plots").mkdir(exist_ok=True)

In [ ]:

def save_single(fn, name):
    fig, ax = plt.subplots(figsize=(6, 4))
    fn(ax)
    fig.tight_layout()
    fig.savefig(f"plots/{name}.png", dpi=130)
    plt.close(fig)

In [ ]:
def class_balance(ax):
    counts = df.cls.value_counts().reindex(classes)
    bars = ax.bar(counts.index, counts.values, color=[colors[c] for c in counts.index])
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height(), int(b.get_height()),
                ha="center", va="bottom", fontsize=9)
    ax.set_title("Class balance"); ax.set_ylabel("images")
 

In [ ]:
def intensity_hist(ax):
    for c in classes:
        ax.hist(df[df.cls == c]["mean"], bins=30, alpha=0.55, label=c, color=colors[c])
    ax.set_title("Mean pixel intensity per image")
    ax.set_xlabel("mean intensity (0-255)"); ax.set_ylabel("count"); ax.legend()

In [ ]:
def intensity_box(ax):
    ax.boxplot([df[df.cls == c]["mean"] for c in classes], labels=classes, showfliers=True)
    ax.set_title("Brightness by class"); ax.set_ylabel("mean intensity")

In [ ]:
def contrast_box(ax):
    ax.boxplot([df[df.cls == c]["std"] for c in classes], labels=classes, showfliers=True)
    ax.set_title("Contrast (pixel std) by class"); ax.set_ylabel("std")

In [ ]:
 
def size_scatter(ax):
    for c in classes:
        g = df[df.cls == c]
        ax.scatter(g.width, g.height, s=12, alpha=0.5, label=c, color=colors[c])
    ax.set_title("Image dimensions"); ax.set_xlabel("width (px)"); ax.set_ylabel("height (px)")
    ax.legend()
 

In [ ]:
def black_border(ax):
    for c in classes:
        ax.hist(df[df.cls == c]["black_frac"], bins=30, alpha=0.55, label=c, color=colors[c])
    ax.set_title("Fraction of near-black pixels (borders/padding)")
    ax.set_xlabel("black fraction"); ax.set_ylabel("count"); ax.legend()

In [ ]:
def aspect_ratio(ax):
    df["_ar"] = df.width / df.height
    for c in classes:
        ax.hist(df[df.cls == c]["_ar"], bins=30, alpha=0.55, label=c, color=colors[c])
    ax.set_title("Aspect ratio (width/height)"); ax.set_xlabel("ratio"); ax.legend()

In [ ]:
 
def mean_vs_std(ax):
    for c in classes:
        g = df[df.cls == c]
        ax.scatter(g["mean"], g["std"], s=10, alpha=0.4, label=c, color=colors[c])
    ax.set_title("Brightness vs contrast (class separation?)")
    ax.set_xlabel("mean"); ax.set_ylabel("std"); ax.legend()

In [ ]:
 
panels = [
    (class_balance, "class_balance"),
    (intensity_hist, "intensity_hist"),
    (intensity_box, "brightness_box"),
    (contrast_box, "contrast_box"),
    (size_scatter, "image_sizes"),
    (black_border, "black_border"),
    (aspect_ratio, "aspect_ratio"),
    (mean_vs_std, "mean_vs_std"),
]
